<a href="https://colab.research.google.com/github/sudharshan-saranathan/pdf-parser/blob/main/Chart_Pipeline_Demo_FullCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Chart Understanding Demo — End-to-end Pipeline

This notebook walks through extracting structured data from research-paper charts.

**Pipeline:**

1. Upload a research-paper PDF → render every page as an image
2. Detect chart regions on each page (YOLO)
3. Classify each chart's type (bar / line / pie / scatter / other) (YOLO)
4. Run OCR (PaddleOCR) on every chart
5. Use Qwen2.5-VL to semantically label each OCR region (title, axis tick, data label, etc.)
6. Run bar/line-specific YOLO detectors and produce a combined overlay + JSON

**How to use:**
- Make sure runtime is set to **GPU** (Runtime → Change runtime type → T4 / L4 / A100)
- 6 sample PDFs ship with the demo bundle in `/content/chart_demo/sample_papers/` if you want to try .
- Run cells **top to bottom**. Each section is one cell.

> **Tip:** if you re-run on a different PDF, just upload it and re-run from Step 3.

## STEP 1 — Install PaddleOCR (4 cells, in order)

PaddleOCR's install is fussy in Colab — these 4 cells are the known-good sequence.

**Run them top to bottom:**
1. First install of PaddlePaddle (CPU) + PaddleOCR
2. **Restart the Colab session** (Runtime → Restart session) after Cell 1 finishes
3. After restart, run Cells 2, 3, 4 in order. Cell 4 prints `PaddleOCR loaded successfully.` and creates the global `ocr` object the rest of the notebook uses.

> Don't merge or skip any of these — the restart between Cell 1 and Cell 2 is required to clear the conflicting wheel state.

In [ ]:
# ==== PADDLEOCR CPU INSTALL : SAFE COLAB BLOCK ====

!python -V
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex || true
!pip install -U pip setuptools wheel

# Official Paddle CPU wheel index shown in PaddleOCR quick start
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Install PaddleOCR after PaddlePaddle
!python -m pip install paddleocr

#RESTART THE SESSION

In [ ]:
# After Restarting the Session Run From Here
#==== CLEAN CONFLICTS ====

!pip uninstall -y \
  langchain \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langsmith \
  paddleocr \
  paddlex \
  paddlepaddle \
  paddlepaddle-gpu

In [ ]:
# ==== INSTALL PADDLE OCR (CPU SAFE) ====

!python -V
!pip install -U pip setuptools wheel

# Official Paddle CPU install
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Then PaddleOCR
!python -m pip install paddleocr==3.4.0

In [ ]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import paddle
print("Paddle version:", paddle.__version__)
print("CUDA:", paddle.is_compiled_with_cuda())

from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("PaddleOCR loaded successfully.")

## STEP 2 — Install the remaining dependencies

Qwen2.5-VL, YOLO (ultralytics), PyMuPDF, gdown. Safe to run after the PaddleOCR session restart.

In [ ]:
# ==== Qwen2.5-VL ====
!pip install -q transformers accelerate qwen-vl-utils

# ==== YOLO + PDF + Drive download ====
!pip install -q ultralytics opencv-python pillow matplotlib pandas tqdm
!pip install -q pymupdf gdown

print("✅ Other dependencies installed")
!nvidia-smi | head -10

## STEP 3 — Download demo bundle (weights + sample papers)

Downloads the model-weights zip from Google Drive and extracts it to `/content/chart_demo/`.
This contains all YOLO weights used by the pipeline plus 5 sample PDF papers in `sample_papers/`.

In [ ]:
from pathlib import Path
import zipfile, shutil

ROOT = Path("/content/chart_demo")
ZIP_PATH = Path("/content/chart_bar_line_qwen_demo_bundle.zip")

# Demo bundle Google Drive ID (replace with your own if you re-package)
DRIVE_FILE_ID = "185upEGS0nI_IjGxoneglgjIfKBGvRNnU"

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

!gdown --id "$DRIVE_FILE_ID" -O "$ZIP_PATH"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(ROOT)

# Normalize any Windows-style backslash paths inside the zip
for p in list(ROOT.rglob("*")):
    if p.is_file() and "\\" in p.name:
        parts = p.name.split("\\")
        new_path = p.parent.joinpath(*parts)
        new_path.parent.mkdir(parents=True, exist_ok=True)
        if not new_path.exists():
            shutil.move(str(p), str(new_path))

print("✅ Extracted to:", ROOT)
for p in sorted(ROOT.iterdir())[:15]:
    print(" -", p.name)

## STEP 4 — Pick a PDF and render its pages

By default this picks up the first `.pdf` it finds under `/content/`.
- To use one of the **sample papers**: change `PDF_PATH` to one of the files under `/content/chart_demo/sample_papers/`.
- To use **your own PDF**: upload it to `/content/` via the sidebar, then re-run this cell.

Pages are rendered at 2x zoom and saved as PNGs.

In [ ]:
from pathlib import Path
import shutil, sys, json, zipfile, signal
import fitz  # PyMuPDF
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/content/chart_demo")
OUT = Path("/content/chart_demo_outputs")
PAGE_OUT_DIR = OUT / "pdf_pages"

# ---------- Download & extract sample papers ----------
SAMPLE_PAPERS_DRIVE_ID = "1m4MgUIP93ds8Jr7lMyp6csSKPPneew5w"

SAMPLE_DIR = ROOT / "sample_papers"
SAMPLE_ZIP = Path("/content/sample_papers.zip")

if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

if SAMPLE_ZIP.exists():
    SAMPLE_ZIP.unlink()

!gdown "$SAMPLE_PAPERS_DRIVE_ID" -O "$SAMPLE_ZIP"

with zipfile.ZipFile(SAMPLE_ZIP, "r") as z:
    z.extractall(SAMPLE_DIR)

samples = sorted(SAMPLE_DIR.rglob("*.pdf"))[:6]
assert samples, "❌ No PDF files found in the sample zip."

# ---------- Colab input box with 90 sec timeout ----------
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

def timed_input_box(prompt, timeout=90, default="1"):
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout)

    try:
        value = input(prompt).strip()
        signal.alarm(0)

        if value == "":
            print(f"⚠️ Empty input. Defaulting to sample [{default}].")
            return default

        return value

    except TimeoutException:
        print(f"\n⏱️ No input within {timeout}s. Defaulting to sample [{default}].")
        return default

    finally:
        signal.alarm(0)

# ---------- Pick one of 6 PDFs ----------
print("\n📄 Available sample papers:")
for i, p in enumerate(samples, 1):
    print(f"  [{i}] {p.name}")

choice = timed_input_box(
    "\n👉 Enter sample number 1-6 (default = 1 after 90 sec): ",
    timeout=90,
    default="1"
)

if choice.isdigit() and 1 <= int(choice) <= len(samples):
    PDF_PATH = samples[int(choice) - 1]
else:
    print("⚠️ Invalid choice. Defaulting to sample [1].")
    PDF_PATH = samples[0]

print(f"\n✅ Using: {PDF_PATH.name}")

# ---------- Locate demo root & weights ----------
if (ROOT / "weights").exists():
    DEMO_ROOT = ROOT
else:
    DEMO_ROOT = next(
        (p for p in ROOT.iterdir() if p.is_dir() and (p / "weights").exists()),
        ROOT
    )

required_weights = {
    "chart_detector":  "chart_detector_v3.pt",
    "plot_classifier": "Plot_Classifier_new_latest.pt",
    "bar_detection":   "Bar_Detection_Yolo_v4.pt",
    "line_axis":       "axis_finder.pt",
    "line_plot_area":  "plot_area_detector_line_v1.pt",
    "line_embed":      "best_line_embed_instance_v1.pt",
}

FOUND = {}
for key, fname in required_weights.items():
    matches = list(DEMO_ROOT.rglob(fname))
    if matches:
        FOUND[key] = str(matches[0])
    else:
        print(f"⚠️ Weight not found: {fname}")

OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "demo_root": str(DEMO_ROOT),
    "out": str(OUT),
    "sample_pdf": str(PDF_PATH),
    **FOUND
}

with open(OUT / "demo_paths.json", "w") as f:
    json.dump(PATHS, f, indent=2)

print("\n✅ Saved paths:", OUT / "demo_paths.json")

# ---------- Render PDF pages to PNG ----------
if PAGE_OUT_DIR.exists():
    shutil.rmtree(PAGE_OUT_DIR)
PAGE_OUT_DIR.mkdir(parents=True, exist_ok=True)

ZOOM = 2.0
MATRIX = fitz.Matrix(ZOOM, ZOOM)

doc = fitz.open(str(PDF_PATH))
page_pngs = []

for i in range(len(doc)):
    pix = doc[i].get_pixmap(matrix=MATRIX, alpha=False)
    out_path = PAGE_OUT_DIR / f"page_{i+1:03d}.png"
    pix.save(str(out_path))
    page_pngs.append(out_path)

doc.close()

print(f"\n✅ Rendered {len(page_pngs)} pages → {PAGE_OUT_DIR}")

plt.figure(figsize=(9, 12))
plt.imshow(Image.open(page_pngs[4]).convert("RGB"))
plt.axis("off")
plt.title(f"First page: {page_pngs[4].name}")
plt.show()

## STEP 5 — Detect chart regions on every page

Runs `chart_detector_v3.pt` (YOLO) on every page PNG. Saves:
- Each detected chart as a cropped PNG → `chart_demo_outputs/chart_crops/`
- Per-page overlay showing detections → `chart_demo_outputs/page_chart_detection_overlays/`
- Metadata CSV with bbox, page number, plot-on-page index

In [ ]:
from pathlib import Path
import cv2, json, shutil
import pandas as pd
from ultralytics import YOLO
from IPython.display import display

OUT = Path("/content/chart_demo_outputs")
PAGE_OUT_DIR = OUT / "pdf_pages"
CHART_CROP_DIR = OUT / "chart_crops"
DETECT_VIS_DIR = OUT / "page_chart_detection_overlays"
META_CSV = OUT / "chart_crops_metadata.csv"

with open(OUT / "demo_paths.json") as f:
    PATHS = json.load(f)

for d in [CHART_CROP_DIR, DETECT_VIS_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

chart_model = YOLO(PATHS["chart_detector"])

page_paths = sorted(PAGE_OUT_DIR.glob("page_*.png"))
records = []
CROP_PAD = 12

for page_idx, page_path in enumerate(page_paths, start=1):
    img_bgr = cv2.imread(str(page_path))
    h, w = img_bgr.shape[:2]

    res = chart_model.predict(str(page_path), conf=0.25, iou=0.45, verbose=False)[0]
    dets = []
    if res.boxes is not None and len(res.boxes) > 0:
        for b in res.boxes:
            x1,y1,x2,y2 = b.xyxy[0].cpu().numpy().tolist()
            dets.append({"xyxy":[x1,y1,x2,y2],
                         "conf": float(b.conf[0].cpu().item()),
                         "cls":  int(b.cls[0].cpu().item()) if b.cls is not None else 0})
    dets.sort(key=lambda d: (d["xyxy"][1], d["xyxy"][0]))  # reading order

    vis = img_bgr.copy()
    for plot_idx, det in enumerate(dets, start=1):
        x1,y1,x2,y2 = det["xyxy"]
        cx1 = max(0, int(x1) - CROP_PAD)
        cy1 = max(0, int(y1) - CROP_PAD)
        cx2 = min(w, int(x2) + CROP_PAD)
        cy2 = min(h, int(y2) + CROP_PAD)

        crop = img_bgr[cy1:cy2, cx1:cx2]
        crop_name = f"page_{page_idx:03d}_plot_{plot_idx:02d}.png"
        crop_path = CHART_CROP_DIR / crop_name
        cv2.imwrite(str(crop_path), crop)

        cv2.rectangle(vis, (cx1,cy1), (cx2,cy2), (0,180,0), 4)
        cv2.putText(vis, f"P{page_idx}-Plot{plot_idx}",
                    (cx1, max(35, cy1-12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,180,0), 3, cv2.LINE_AA)

        records.append({
            "page_number": page_idx,
            "plot_number_on_page": plot_idx,
            "global_plot_id": len(records) + 1,
            "crop_filename": crop_name,
            "crop_path": str(crop_path),
            "bbox_page_xyxy": [cx1,cy1,cx2,cy2],
            "detector_confidence": round(det["conf"], 4),
        })

    cv2.imwrite(str(DETECT_VIS_DIR / f"page_{page_idx:03d}_detections.png"), vis)
    print(f"Page {page_idx:03d}: {len(dets)} chart(s)")

df = pd.DataFrame(records)
df.to_csv(META_CSV, index=False)

print(f"\n✅ {len(records)} chart crops saved to: {CHART_CROP_DIR}")
display(df.head(15))

## STEP 6 — Classify each chart's type

Runs `Plot_Classifier_new_latest.pt` on every chart crop to predict bar / line / pie / scatter / other.

In [ ]:
from pathlib import Path
import json, math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from IPython.display import display

OUT = Path("/content/chart_demo_outputs")
META_CSV = OUT / "chart_crops_metadata.csv"
CLASSIFIED_CSV = OUT / "chart_crops_classified_metadata.csv"

with open(OUT / "demo_paths.json") as f:
    PATHS = json.load(f)

clf_model = YOLO(PATHS["plot_classifier"])
print("Classifier classes:", clf_model.names)

df = pd.read_csv(META_CSV)
pred_rows = []

for _, row in df.iterrows():
    crop_path = Path(row["crop_path"])
    r = clf_model.predict(str(crop_path), verbose=False)[0]

    if hasattr(r, "probs") and r.probs is not None:
        top1_id   = int(r.probs.top1)
        top1_conf = float(r.probs.top1conf.cpu().item())
        pred_name = clf_model.names.get(top1_id, str(top1_id))
    else:
        pred_name, top1_id, top1_conf = "unknown", -1, 0.0

    rec = row.to_dict()
    rec["pred_chart_type"] = pred_name
    rec["pred_chart_type_confidence"] = round(top1_conf, 4)
    pred_rows.append(rec)

classified_df = pd.DataFrame(pred_rows)
classified_df.to_csv(CLASSIFIED_CSV, index=False)

print("\n=== Chart type counts ===")
display(classified_df.groupby("pred_chart_type").size().reset_index(name="count"))

print("\n=== All charts ===")
display(classified_df[[
    "global_plot_id", "page_number", "plot_number_on_page",
    "crop_filename", "pred_chart_type", "pred_chart_type_confidence",
]])

# ---------- Gallery preview ----------
MAX_SHOW = min(12, len(classified_df))
cols = 3
rows = math.ceil(MAX_SHOW / cols)
plt.figure(figsize=(cols * 5, rows * 4))
for i in range(MAX_SHOW):
    row = classified_df.iloc[i]
    img = Image.open(row["crop_path"]).convert("RGB")
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img); plt.axis("off")
    plt.title(f"P{row['page_number']} Plot{row['plot_number_on_page']}\n"
              f"{row['pred_chart_type']} ({row['pred_chart_type_confidence']:.2f})",
              fontsize=9)
plt.tight_layout(); plt.show()

## STEP 7 — OCR helpers (PaddleOCR already loaded in STEP 1)

Defines:
- `run_paddle_ocr(path)` → list of text regions with bbox & confidence
- `featurize_box(box, W, H)` → adds position bands, numeric/percent flags, rotation
- role lists for bar / line / pie / scatter charts

The `ocr` PaddleOCR object was already created in STEP 1, Cell 4.

In [ ]:
import math, re, json
from pathlib import Path
import numpy as np
from PIL import Image

# (PaddleOCR `ocr` object was already created in STEP 1, Cell 4)

# ---------- OCR runner ----------
def run_paddle_ocr(image_path):
    "Return list of {text, bbox, poly, confidence, angle}."
    result = ocr.predict(input=str(image_path))
    boxes = []
    for res in result:
        try:
            texts  = res["rec_texts"]
            scores = res["rec_scores"]
            polys  = res["rec_polys"]
        except (KeyError, TypeError):
            data = res.json.get("res", res.json) if hasattr(res, "json") else res
            texts, scores, polys = data["rec_texts"], data["rec_scores"], data["rec_polys"]

        for text, score, poly in zip(texts, scores, polys):
            if str(text).strip() == "":
                continue
            poly_arr = np.array(poly, dtype=float)
            x1, y1 = poly_arr.min(axis=0)
            x2, y2 = poly_arr.max(axis=0)
            dx = poly_arr[1][0] - poly_arr[0][0]
            dy = poly_arr[1][1] - poly_arr[0][1]
            angle = math.degrees(math.atan2(dy, dx))
            boxes.append({
                "text": str(text).strip(),
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "poly": poly_arr.tolist(),
                "confidence": float(score),
                "angle": float(angle),
            })
    return boxes

# ---------- Per-box features (used by the role classifier prompt) ----------
NUMERIC_RE  = re.compile(r"^-?\d+(\.\d+)?$")
PERCENT_RE  = re.compile(r"^-?\d+(\.\d+)?%$")
CURRENCY_RE = re.compile(r"[\$€£¥₹]")
SUFFIX_RE   = re.compile(r"^-?\d+(\.\d+)?\s*[kmbKMB]$")
YEAR_RE     = re.compile(r"^'?\d{2,4}$")

def featurize_box(b, W, H):
    t = b["text"]
    cx = (b["bbox"][0] + b["bbox"][2]) / 2
    cy = (b["bbox"][1] + b["bbox"][3]) / 2
    bw = b["bbox"][2] - b["bbox"][0]
    bh = b["bbox"][3] - b["bbox"][1]

    yfrac = cy / max(H, 1)
    xfrac = cx / max(W, 1)
    y_band = ("very_top" if yfrac<0.12 else "top" if yfrac<0.30 else
              "middle" if yfrac<0.70 else "bottom" if yfrac<0.88 else "very_bottom")
    x_band = ("very_left" if xfrac<0.15 else "left" if xfrac<0.35 else
              "center" if xfrac<0.65 else "right" if xfrac<0.85 else "very_right")

    is_rotated = (30 < abs(b["angle"]) < 150) or (bh > bw*1.5 and len(t) > 3)
    clean_num = t.replace(",", "").replace("\u20b9", "").replace("$", "")

    b["features"] = {
        "is_numeric": bool(NUMERIC_RE.match(clean_num) or SUFFIX_RE.match(clean_num)),
        "is_percent": bool(PERCENT_RE.match(clean_num)),
        "is_year":    bool(YEAR_RE.match(t)),
        "is_currency":bool(CURRENCY_RE.search(t)),
        "word_count": len(t.split()),
        "char_count": len(t),
        "y_band": y_band, "x_band": x_band,
        "is_rotated": is_rotated,
    }
    return b

# ---------- Role taxonomies ----------
ROLES_AXIS = ["title","subtitle","x_tick","y_tick","x_axis_title","y_axis_title",
              "data_label","legend","caption","other"]
ROLES_PIE  = ["title","subtitle","slice_label","slice_value","legend","caption","other"]

def roles_for_type(chart_type):
    return ROLES_PIE if chart_type == "pie" else ROLES_AXIS

print("✅ OCR + helpers defined")

## STEP 8 — Load Qwen2.5-VL (one clean copy, fully on GPU)

Loads **Qwen/Qwen2.5-VL-7B-Instruct** with `device_map="cuda:0"` so the whole model stays on GPU.
(Using `device_map="auto"` is risky — if VRAM is tight it silently CPU-offloads layers,
making each inference 20-50x slower.)

If you have a T4 (15 GB) and hit OOM, swap the model name to `Qwen/Qwen2.5-VL-3B-Instruct`.

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
# ============================================================
# STEP 8: Safe Qwen2.5-VL loading for Colab
# T4: Qwen 3B 4-bit
# L4/A100: Qwen 7B bf16, or change FORCE_3B=True for safety
# ============================================================

import os, gc, time

# Must be set before heavy CUDA allocations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

# Clean any old models if cell is re-run
for name in [
    "qwen_model", "qwen_processor",
    "chart_model", "clf_model",
    "bar_axis_model", "bar_det_model", "bar_yolo8_model",
    "line_axis_model", "line_plot_area_model",
]:
    if name in globals():
        print("Deleting:", name)
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.is_available(), (
    "❌ No GPU detected. In Colab: Runtime → Change runtime type → GPU."
)

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name} ({total_gb:.1f} GB)")
!nvidia-smi

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

# For live demo, safer to keep 3B even on L4.
FORCE_3B = True

if FORCE_3B or total_gb < 20:
    MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
    USE_4BIT = True
    print("→ Using Qwen 3B in 4-bit mode")
else:
    MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
    USE_4BIT = False
    print("→ Using Qwen 7B bf16")

print("Loading:", MODEL_NAME)

if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

# IMPORTANT:
# use_fast=False avoids the Qwen processor fast-import error.
qwen_processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    min_pixels=256 * 28 * 28,
    max_pixels=640 * 28 * 28,
)

qwen_model.eval()

print("✅ Qwen model loaded:", MODEL_NAME)
print("Main device:", next(qwen_model.parameters()).device)
!nvidia-smi


def qwen_infer(image, prompt, max_new_tokens=256):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        gen = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    trimmed = gen[0][inputs.input_ids.shape[1]:]
    return qwen_processor.decode(trimmed, skip_special_tokens=True)


# Warmup test
from PIL import Image

t0 = time.time()

reply = qwen_infer(
    Image.new("RGB", (384, 384), "white"),
    "Reply with one word: ok.",
    max_new_tokens=5,
)

print("Warmup reply:", reply)
print(f"Warmup time: {time.time() - t0:.2f}s")

## STEP 9 — Define the role classifier (`classify_roles_fast`)

Sends the chart image + a compact list of OCR regions to Qwen and asks it to label each region
as `title`, `x_tick`, `y_tick`, `data_label`, etc. Returns the boxes annotated with `role`.

In [ ]:
import re, json
from PIL import Image

FAST_ROLE_PROMPT_TEMPLATE = """Classify OCR text regions in this {chart_type} chart.

Valid roles:
{role_list}

OCR regions:
{regions_block}

Rules:
top main text=title; left stacked numbers=y_tick; bottom labels=x_tick;
rotated left text=y_axis_title; bottom axis name=x_axis_title;
numbers on bars/points=data_label; legend entries=legend;
source/note/footer=caption; unclear=other.

Return ONLY compact JSON:
[{{"id":0,"role":"title","confidence":"high"}}]
One item per OCR region."""

def build_regions_block_fast(boxes, max_boxes=18):
    lines = []
    for i, b in enumerate(boxes[:max_boxes]):
        f = b["features"]
        t = b["text"]
        if len(t) > 50:
            t = t[:50] + "..."
        tags = []
        if f["is_numeric"]: tags.append("numeric")
        if f["is_percent"]: tags.append("percent")
        if f["is_year"]:    tags.append("year")
        if f["is_rotated"]: tags.append("rotated")
        tag_str = ",".join(tags) if tags else "text"
        lines.append(f'id={i} text="{t}" pos=({f["x_band"]},{f["y_band"]}) type={tag_str}')
    return "\n".join(lines)

def classify_roles_fast(image_path, boxes, chart_type):
    if not boxes:
        return []
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > 640:
        img.thumbnail((640, 640), Image.LANCZOS)

    prompt = FAST_ROLE_PROMPT_TEMPLATE.format(
        chart_type=chart_type,
        role_list=", ".join(roles_for_type(chart_type)),
        regions_block=build_regions_block_fast(boxes),
    )
    raw = qwen_infer(img, prompt, max_new_tokens=256)

    parsed = []
    try:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
    except Exception as e:
        print("  JSON parse failed:", e)

    id_to_role = {}
    for item in parsed:
        if isinstance(item, dict) and "id" in item:
            id_to_role[int(item["id"])] = (
                item.get("role", "other"),
                item.get("confidence", "low"),
            )

    allowed = set(roles_for_type(chart_type))
    for i, b in enumerate(boxes):
        role, conf = id_to_role.get(i, ("other", "low"))
        if role not in allowed:
            role = "other"
        b["role"] = role
        b["role_confidence"] = conf
    return boxes

print("✅ classify_roles_fast defined")

## STEP 10 — Load the bar & line YOLO detectors

Three small YOLO weights:
- `Bar_Detection_Yolo_v4.pt` — bar bounding boxes
- `plot_area_detector_line_v1.pt` — plot region for line charts
- `axis_finder.pt` — axis regions for line charts

These are quick (a few hundred MB each) and run on GPU.

In [ ]:
from pathlib import Path
import json
from ultralytics import YOLO

with open("/content/chart_demo_outputs/demo_paths.json") as f:
    PATHS = json.load(f)

print("Loading bar detector...")
bar_det_model = YOLO(PATHS["bar_detection"])
print("  \u2713", PATHS["bar_detection"])

print("Loading line plot-area detector...")
line_plot_area_model = YOLO(PATHS["line_plot_area"])
print("  \u2713", PATHS["line_plot_area"])

print("Loading line axis detector...")
line_axis_model = YOLO(PATHS["line_axis"])
print("  \u2713", PATHS["line_axis"])

print("\n\u2705 All YOLO bar/line models loaded")

## STEP 11 — Combined bar + line demo

For every detected bar or line chart:

1. PaddleOCR finds text regions
2. Qwen labels each region (title, ticks, axis titles, data labels, ...)
3. Per-type YOLO runs:
   - **Bar charts** → `Bar_Detection_Yolo_v4` (bars)
   - **Line charts** → plot-area box + axis boxes
4. A combined overlay PNG is saved + a structured JSON entry is appended

Output:
- `bar_line_demo_outputs/reports/*.png` — one image per chart
- `bar_line_demo_outputs/bar_line_demo_results.json` — all results

In [ ]:
# ============================================================
# UPDATED STEP 11: Clean Bar + Line Demo
# - Bar: uses ONLY Bar_Detection_Yolo_v4.pt, bright visible overlay
# - Line: uses plot_area_detector_line_v1.pt + best_line_embed_instance_v1.pt
# - Axis boxes are summarized, not drawn, to avoid 100+ tick clutter
# - Qwen/PaddleOCR text role boxes included
# - Saves clean JSON + report images
# ============================================================

!pip install -q scikit-image scikit-learn

from pathlib import Path
import os, json, math, gc, traceback
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from PIL import Image
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from skimage.morphology import skeletonize

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
OUT = Path("/content/chart_demo_outputs")
PATH_CONFIG = OUT / "demo_paths.json"
CLASSIFIED_CSV = OUT / "chart_crops_classified_metadata.csv"

assert PATH_CONFIG.exists(), "Run CODE 4 first. Missing demo_paths.json"
assert CLASSIFIED_CSV.exists(), "Run CODE 6 first. Missing classified metadata."

with open(PATH_CONFIG, "r") as f:
    PATHS = json.load(f)

DEMO_ROOT = Path(PATHS["demo_root"])

DEMO_OUT = OUT / "bar_line_demo_outputs_final"
DEMO_REPORT_DIR = DEMO_OUT / "reports"
DEMO_OUT.mkdir(parents=True, exist_ok=True)
DEMO_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Required helper availability check
# ------------------------------------------------------------
for fn in ["run_paddle_ocr", "featurize_box", "classify_roles_fast"]:
    assert fn in globals(), f"Missing function: {fn}. Run OCR/Qwen helper cells first."

# ------------------------------------------------------------
# Find weights
# ------------------------------------------------------------
def find_weight(filename):
    matches = list(DEMO_ROOT.rglob(filename))
    assert matches, f"Missing weight: {filename}"
    return matches[0]

BAR_DET_WEIGHT = Path(PATHS.get("bar_detection", "")) if PATHS.get("bar_detection") else find_weight("Bar_Detection_Yolo_v4.pt")

LINE_AXIS_WEIGHT = Path(PATHS.get("line_axis", "")) if PATHS.get("line_axis") else find_weight("axis_finder.pt")
LINE_PLOT_AREA_WEIGHT = Path(PATHS.get("line_plot_area", "")) if PATHS.get("line_plot_area") else find_weight("plot_area_detector_line_v1.pt")
LINE_EMBED_WEIGHT = Path(PATHS.get("line_embed", "")) if PATHS.get("line_embed") else find_weight("best_line_embed_instance_v1.pt")

print("Using bar weight:", BAR_DET_WEIGHT)
print("Using line plot-area weight:", LINE_PLOT_AREA_WEIGHT)
print("Using line axis weight:", LINE_AXIS_WEIGHT)
print("Using line UNet/embed weight:", LINE_EMBED_WEIGHT)

# ------------------------------------------------------------
# Load YOLO models if needed
# ------------------------------------------------------------
if "bar_det_model" not in globals():
    bar_det_model = YOLO(str(BAR_DET_WEIGHT))
    print("Loaded bar_det_model")

if "line_plot_area_model" not in globals():
    line_plot_area_model = YOLO(str(LINE_PLOT_AREA_WEIGHT))
    print("Loaded line_plot_area_model")

if "line_axis_model" not in globals():
    line_axis_model = YOLO(str(LINE_AXIS_WEIGHT))
    print("Loaded line_axis_model")

# ------------------------------------------------------------
# Load actual line UNet model from best_line_embed_instance_v1.pt
# ------------------------------------------------------------
EMBED_DIM = 8

class LineConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class LineUNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        self.e1 = LineConvBlock(3, 32)
        self.e2 = LineConvBlock(32, 64)
        self.e3 = LineConvBlock(64, 128)
        self.e4 = LineConvBlock(128, 256)

        self.pool = nn.MaxPool2d(2)
        self.b = LineConvBlock(256, 512)

        self.u4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.d4 = LineConvBlock(512, 256)

        self.u3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.d3 = LineConvBlock(256, 128)

        self.u2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d2 = LineConvBlock(128, 64)

        self.u1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d1 = LineConvBlock(64, 32)

        self.head_mask = nn.Conv2d(32, 1, 1)
        self.head_tangent = nn.Conv2d(32, 2, 1)
        self.head_embed = nn.Conv2d(32, EMBED_DIM, 1)
        self.head_style = nn.Conv2d(32, 5, 1)
        self.head_marker = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))

        b = self.b(self.pool(e4))

        x = self.u4(b)
        if x.shape[-2:] != e4.shape[-2:]:
            x = F.interpolate(x, size=e4.shape[-2:], mode="bilinear", align_corners=False)
        x = self.d4(torch.cat([x, e4], dim=1))

        x = self.u3(x)
        if x.shape[-2:] != e3.shape[-2:]:
            x = F.interpolate(x, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        x = self.d3(torch.cat([x, e3], dim=1))

        x = self.u2(x)
        if x.shape[-2:] != e2.shape[-2:]:
            x = F.interpolate(x, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        x = self.d2(torch.cat([x, e2], dim=1))

        x = self.u1(x)
        if x.shape[-2:] != e1.shape[-2:]:
            x = F.interpolate(x, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        x = self.d1(torch.cat([x, e1], dim=1))

        return {
            "mask": self.head_mask(x),
            "tangent": self.head_tangent(x),
            "embed": self.head_embed(x),
            "style": self.head_style(x),
            "marker": self.head_marker(x),
        }

LINE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if "line_unet_model" not in globals():
    try:
        line_unet_model = LineUNetV2().to(LINE_DEVICE)
        ckpt = torch.load(str(LINE_EMBED_WEIGHT), map_location=LINE_DEVICE)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        state = {k.replace("module.", ""): v for k, v in state.items()}
        missing, unexpected = line_unet_model.load_state_dict(state, strict=False)
        line_unet_model.eval()
        print("Loaded line_unet_model on", LINE_DEVICE)
        print("Missing:", len(missing), "Unexpected:", len(unexpected))
    except RuntimeError as e:
        print("GPU load failed, falling back to CPU:", e)
        LINE_DEVICE = torch.device("cpu")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        line_unet_model = LineUNetV2().to(LINE_DEVICE)
        ckpt = torch.load(str(LINE_EMBED_WEIGHT), map_location=LINE_DEVICE)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        state = {k.replace("module.", ""): v for k, v in state.items()}
        missing, unexpected = line_unet_model.load_state_dict(state, strict=False)
        line_unet_model.eval()
        print("Loaded line_unet_model on CPU")

# ------------------------------------------------------------
# Filter bar + line crops
# ------------------------------------------------------------
classified_df = pd.read_csv(CLASSIFIED_CSV)

def norm_chart_type(x):
    x = str(x).lower()
    if "bar" in x:
        return "bar"
    if "line" in x:
        return "line"
    if "pie" in x or "donut" in x:
        return "pie"
    if "scatter" in x:
        return "scatter"
    return "other"

classified_df["normalized_chart_type"] = classified_df["pred_chart_type"].apply(norm_chart_type)

target_df = classified_df[
    classified_df["normalized_chart_type"].isin(["bar", "line"])
].copy()

print(
    f"{len(target_df)} bar+line crops "
    f"({(target_df.normalized_chart_type == 'bar').sum()} bar, "
    f"{(target_df.normalized_chart_type == 'line').sum()} line)"
)

display(target_df[[
    "global_plot_id",
    "page_number",
    "plot_number_on_page",
    "crop_filename",
    "pred_chart_type",
    "pred_chart_type_confidence",
    "normalized_chart_type",
]])

# ------------------------------------------------------------
# YOLO helpers
# ------------------------------------------------------------
def yolo_boxes(model, image_path_or_arr, conf=0.25, iou=0.45):
    res = model.predict(image_path_or_arr, conf=conf, iou=iou, verbose=False)[0]
    out = []

    if res.boxes is None or len(res.boxes) == 0:
        return out

    xyxy = res.boxes.xyxy.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    clss = res.boxes.cls.cpu().numpy().astype(int)
    names = res.names

    for (x1, y1, x2, y2), c, k in zip(xyxy, confs, clss):
        label = names.get(k, str(k)) if isinstance(names, dict) else str(k)
        out.append({
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "conf": round(float(c), 4),
            "class": str(label),
        })

    return out

def fallback_plot_box(img_bgr):
    H, W = img_bgr.shape[:2]
    return [int(W * 0.10), int(H * 0.12), int(W * 0.94), int(H * 0.82)]

def detect_line_plot_area(img_bgr):
    boxes = yolo_boxes(line_plot_area_model, img_bgr, conf=0.08, iou=0.45)

    if not boxes:
        return fallback_plot_box(img_bgr), 0.0, "fallback"

    # largest/highest-confidence box
    best = max(
        boxes,
        key=lambda b: (b["conf"], (b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1]))
    )

    x1, y1, x2, y2 = best["bbox"]
    H, W = img_bgr.shape[:2]

    pad_x = int((x2 - x1) * 0.01)
    pad_y = int((y2 - y1) * 0.01)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(W - 1, x2 + pad_x)
    y2 = min(H - 1, y2 + pad_y)

    return [x1, y1, x2, y2], best["conf"], "yolo"

def detect_bar_boxes(crop_path):
    bars = yolo_boxes(bar_det_model, str(crop_path), conf=0.18, iou=0.45)
    return {
        "bars": bars,
        "summary": {
            "bar_boxes": len(bars),
        }
    }

# ------------------------------------------------------------
# Actual line UNet inference
# ------------------------------------------------------------
def pad_to_multiple(img, mult=16):
    h, w = img.shape[:2]
    nh = int(math.ceil(h / mult) * mult)
    nw = int(math.ceil(w / mult) * mult)

    ph = nh - h
    pw = nw - w

    padded = cv2.copyMakeBorder(
        img,
        0, ph,
        0, pw,
        cv2.BORDER_CONSTANT,
        value=(255, 255, 255)
    )

    return padded, (h, w)

def run_line_unet_mask(img_bgr, plot_box, prob_th=0.40, marker_th=0.55, min_area=45):
    H, W = img_bgr.shape[:2]
    x1, y1, x2, y2 = plot_box

    keep = np.zeros((H, W), dtype=np.uint8)
    keep[y1:y2, x1:x2] = 1

    img_pad, (orig_h, orig_w) = pad_to_multiple(img_bgr, 16)

    inp = img_pad.astype(np.float32) / 255.0
    inp = torch.from_numpy(inp.transpose(2, 0, 1)).unsqueeze(0).float().to(LINE_DEVICE)

    with torch.inference_mode():
        out = line_unet_model(inp)

    prob = torch.sigmoid(out["mask"])[0, 0].detach().cpu().numpy()[:orig_h, :orig_w]
    marker = torch.sigmoid(out["marker"])[0, 0].detach().cpu().numpy()[:orig_h, :orig_w]

    raw_mask = (prob > prob_th).astype(np.uint8)
    marker_mask = (marker > marker_th).astype(np.uint8)

    mask = np.maximum(raw_mask, marker_mask)
    mask = mask * keep

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    clean = np.zeros_like(mask)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            clean[labels == i] = 1

    skel = skeletonize(clean > 0).astype(np.uint8)

    n_components, comp_labels, comp_stats, _ = cv2.connectedComponentsWithStats(skel, connectivity=8)

    series_components = []
    for i in range(1, n_components):
        area = int(comp_stats[i, cv2.CC_STAT_AREA])
        if area < 12:
            continue
        ys, xs = np.where(comp_labels == i)
        order = np.argsort(xs)
        series_components.append({
            "series_id": len(series_components) + 1,
            "points": int(len(xs)),
            "bbox": [
                int(xs.min()), int(ys.min()),
                int(xs.max()), int(ys.max())
            ],
            "xs": xs[order],
            "ys": ys[order],
        })

    return prob, clean, skel, series_components

def detect_line_full(crop_path):
    img_bgr = cv2.imread(str(crop_path))
    assert img_bgr is not None, f"Could not read {crop_path}"

    plot_box, plot_conf, plot_source = detect_line_plot_area(img_bgr)

    # Axis model only for counts, not overlay clutter
    axis_raw = yolo_boxes(line_axis_model, img_bgr, conf=0.08, iou=0.45)
    axis_class_counts = Counter([b["class"] for b in axis_raw])

    prob, clean_mask, skel, series_components = run_line_unet_mask(
        img_bgr,
        plot_box,
        prob_th=0.40,
        marker_th=0.55,
        min_area=45
    )

    return {
        "plot_area": [{
            "bbox": plot_box,
            "conf": round(float(plot_conf), 4),
            "source": plot_source,
        }],
        "axis_raw_count": len(axis_raw),
        "axis_class_counts": dict(axis_class_counts),
        "line_mask_pixels": int(clean_mask.sum()),
        "line_skeleton_pixels": int(skel.sum()),
        "series_components": [
            {
                "series_id": s["series_id"],
                "points": s["points"],
                "bbox": s["bbox"],
            }
            for s in series_components
        ],
        "summary": {
            "plot_area_boxes": 1,
            "axis_raw_boxes_hidden": len(axis_raw),
            "line_mask_pixels": int(clean_mask.sum()),
            "line_skeleton_pixels": int(skel.sum()),
            "series_components": len(series_components),
        },
        "_clean_mask": clean_mask,
        "_skeleton": skel,
        "_series_components_full": series_components,
    }

# ------------------------------------------------------------
# Text grouping + clean JSON
# ------------------------------------------------------------
ROLE_ORDER = [
    "title", "subtitle", "y_axis_title", "x_axis_title",
    "y_tick", "x_tick", "data_label", "legend", "caption", "other"
]

ROLE_COLORS = {
    "title": "#E53935",
    "subtitle": "#FB8C00",
    "x_tick": "#43A047",
    "y_tick": "#1E88E5",
    "x_axis_title": "#6A1B9A",
    "y_axis_title": "#8E24AA",
    "data_label": "#D81B60",
    "legend": "#00ACC1",
    "caption": "#5D4037",
    "other": "#9E9E9E",
}

BAR_FILL = "#FFFF00"
BAR_EDGE = "#FF00FF"
PLOT_AREA_COLOR = "#00E5FF"
LINE_MASK_COLOR = "#FF1744"
LINE_SKEL_COLOR = "#00FF00"

def group_text_regions(text_regions):
    grouped = defaultdict(list)
    for r in text_regions:
        grouped[r["role"]].append(r["text"])
    return dict(grouped)

def build_clean_summary(result):
    grouped = group_text_regions(result.get("text_regions", []))

    lines = [
        f"{result['crop']}   [{result['chart_type']}]",
        "-" * 72
    ]

    for role in ROLE_ORDER:
        vals = grouped.get(role, [])
        if not vals:
            continue

        label = role.upper().ljust(14)
        joined = " | ".join(vals)

        if len(joined) <= 95:
            lines.append(f"  {label}  {joined}")
        else:
            lines.append(f"  {label}  {vals[0]}")
            for v in vals[1:]:
                lines.append(f"  {' ' * 14}  {v}")

    lines.append("")
    det = result.get("detections", {})

    if result["chart_type"] == "bar":
        bars = det.get("bars", [])
        lines.append(f"  YOLO BARS      {len(bars)} detected")

        for i, b in enumerate(bars[:12], 1):
            x1, y1, x2, y2 = b["bbox"]
            lines.append(
                f"    [{i}] bbox=({x1},{y1}) to ({x2},{y2}) "
                f"size={x2-x1}x{y2-y1} conf={b['conf']:.2f}"
            )

        if len(bars) > 12:
            lines.append(f"    ... {len(bars) - 12} more bar boxes hidden in text panel")

    else:
        pa = det.get("plot_area", [])
        lines.append(f"  PLOT AREA      {len(pa)} detected")
        for i, b in enumerate(pa, 1):
            x1, y1, x2, y2 = b["bbox"]
            lines.append(
                f"    [{i}] bbox=({x1},{y1}) to ({x2},{y2}) "
                f"conf={b['conf']:.2f} source={b['source']}"
            )

        lines.append(f"  LINE MASK      pixels={det.get('line_mask_pixels', 0)}")
        lines.append(f"  LINE SKELETON  pixels={det.get('line_skeleton_pixels', 0)}")
        lines.append(f"  SERIES PARTS   {len(det.get('series_components', []))}")

        for s in det.get("series_components", [])[:8]:
            lines.append(f"    series {s['series_id']}: points={s['points']} bbox={s['bbox']}")

        lines.append(f"  AXIS MODEL     raw={det.get('axis_raw_count', 0)} hidden/summarized")
        counts = det.get("axis_class_counts", {})
        if counts:
            lines.append("                 " + ", ".join([f"{k}:{v}" for k, v in counts.items()]))

    return "\n".join(lines)

def make_clean_json_result(result):
    text_regions = result.get("text_regions", [])
    grouped_text = group_text_regions(text_regions)
    det = result.get("detections", {})

    clean = {
        "crop": result["crop"],
        "chart_type": result["chart_type"],
        "global_plot_id": result["global_plot_id"],
        "page_number": result["page_number"],
        "plot_number_on_page": result["plot_number_on_page"],
        "text_summary": grouped_text,
        "text_regions": text_regions,
        "detection_summary": det.get("summary", {}),
        "report_path": result.get("report_path", ""),
    }

    if result["chart_type"] == "bar":
        clean["detections"] = {
            "bars": det.get("bars", []),
        }
    else:
        clean["detections"] = {
            "plot_area": det.get("plot_area", []),
            "line_mask_pixels": det.get("line_mask_pixels", 0),
            "line_skeleton_pixels": det.get("line_skeleton_pixels", 0),
            "series_components": det.get("series_components", []),
            "axis_raw_count": det.get("axis_raw_count", 0),
            "axis_class_counts": det.get("axis_class_counts", {}),
            "note": "axis/tick detections are summarized, not drawn, to keep overlay readable",
        }

    return clean

# ------------------------------------------------------------
# Overlay drawing
# ------------------------------------------------------------
def draw_overlay_final(crop_path, result, save_path):
    img = Image.open(crop_path).convert("RGB")
    img_np = np.array(img)
    summary = build_clean_summary(result)
    n_lines = summary.count("\n") + 1

    fig = plt.figure(figsize=(12, 6 + n_lines * 0.16))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, max(0.35, n_lines * 0.035)])

    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(img)
    ax_img.axis("off")

    det = result.get("detections", {})
    extra_handles = []

    if result["chart_type"] == "bar":
        bars = det.get("bars", [])

        for b in bars:
            x1, y1, x2, y2 = b["bbox"]
            ax_img.add_patch(mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=3.2,
                edgecolor=BAR_EDGE,
                facecolor=BAR_FILL,
                alpha=0.42
            ))

        extra_handles.append(
            mpatches.Patch(color=BAR_FILL, label=f"Bar_Detection_Yolo_v4 ({len(bars)})")
        )

    else:
        plot_area = det.get("plot_area", [])
        clean_mask = det.get("_clean_mask", None)
        skel = det.get("_skeleton", None)

        # draw red translucent line mask
        if clean_mask is not None and clean_mask.sum() > 0:
            line_overlay = np.zeros_like(img_np, dtype=np.float32)
            line_overlay[:, :, 0] = 255
            mask = clean_mask > 0
            blended = img_np.astype(np.float32)
            blended[mask] = 0.45 * line_overlay[mask] + 0.55 * blended[mask]
            ax_img.imshow(np.clip(blended, 0, 255).astype(np.uint8))

        # draw green skeleton points
        if skel is not None and skel.sum() > 0:
            ys, xs = np.where(skel > 0)
            ax_img.scatter(xs, ys, s=4, c=LINE_SKEL_COLOR, alpha=0.85)

        # plot area box
        for b in plot_area:
            x1, y1, x2, y2 = b["bbox"]
            ax_img.add_patch(mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=4.0,
                edgecolor=PLOT_AREA_COLOR,
                facecolor="none",
                alpha=1.0
            ))

        extra_handles.append(
            mpatches.Patch(color=PLOT_AREA_COLOR, label=f"plot_area ({len(plot_area)})")
        )
        extra_handles.append(
            mpatches.Patch(color=LINE_MASK_COLOR, label="line mask")
        )
        extra_handles.append(
            mpatches.Patch(color=LINE_SKEL_COLOR, label="line skeleton")
        )

    # Qwen/OCR role boxes
    for r in result.get("text_regions", []):
        color = ROLE_COLORS.get(r["role"], "#9E9E9E")
        x1, y1, x2, y2 = r["bbox"]

        ax_img.add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.4,
            edgecolor=color,
            facecolor="none"
        ))

    role_handles = [
        mpatches.Patch(color=c, label=r)
        for r, c in ROLE_COLORS.items()
        if any(tr.get("role") == r for tr in result.get("text_regions", []))
    ]

    handles = role_handles + extra_handles
    if handles:
        ax_img.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.92)

    n_txt = len(result.get("text_regions", []))

    if result["chart_type"] == "bar":
        head = f"{result['crop']} | bar | text_regions={n_txt} | bars={len(det.get('bars', []))}"
    else:
        head = (
            f"{result['crop']} | line | text_regions={n_txt} | "
            f"plot_area={len(det.get('plot_area', []))} | "
            f"line_pixels={det.get('line_mask_pixels', 0)} | "
            f"series_parts={len(det.get('series_components', []))}"
        )

    ax_img.set_title(head, fontsize=10, fontweight="bold")

    ax_txt = fig.add_subplot(gs[1])
    ax_txt.axis("off")
    ax_txt.text(
        0.02, 0.98,
        summary,
        family="monospace",
        fontsize=8.5,
        verticalalignment="top",
        horizontalalignment="left"
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()

# ------------------------------------------------------------
# Main loop
# ------------------------------------------------------------
all_results = []
clean_results = []

for _, row in tqdm(target_df.iterrows(), total=len(target_df), desc="final bar+line"):
    crop_path = Path(row["crop_path"])
    chart_type = row["normalized_chart_type"]

    print(f"\n--- {crop_path.name} [{chart_type}] ---")

    try:
        W, H = Image.open(crop_path).size

        boxes = run_paddle_ocr(crop_path)
        for b in boxes:
            featurize_box(b, W, H)

        print(f"  OCR: {len(boxes)} regions")

        boxes = classify_roles_fast(crop_path, boxes, chart_type)
        print("  Qwen: roles assigned")

        if chart_type == "bar":
            detections = detect_bar_boxes(crop_path)
            print(f"  BAR YOLO: {len(detections['bars'])} bars")
        else:
            detections = detect_line_full(crop_path)
            print(
                f"  LINE: plot_area={len(detections['plot_area'])}, "
                f"line_pixels={detections['line_mask_pixels']}, "
                f"series_parts={len(detections['series_components'])}, "
                f"axis_raw_hidden={detections['axis_raw_count']}"
            )

        result = {
            "crop": crop_path.name,
            "chart_type": chart_type,
            "global_plot_id": int(row["global_plot_id"]),
            "page_number": int(row["page_number"]),
            "plot_number_on_page": int(row["plot_number_on_page"]),
            "text_regions": [
                {
                    "role": b.get("role", "other"),
                    "text": b["text"],
                    "bbox": b["bbox"],
                    "ocr_conf": round(float(b.get("confidence", 0)), 4),
                }
                for b in boxes
            ],
            "detections": detections,
        }

        save_path = DEMO_REPORT_DIR / f"{crop_path.stem}_final_demo.png"
        draw_overlay_final(crop_path, result, save_path)

        result["report_path"] = str(save_path)

        # Store raw in memory; clean JSON strips mask arrays
        all_results.append(result)
        clean_results.append(make_clean_json_result(result))

        print(f"  saved {save_path.name}")

    except Exception as e:
        print(f"  FAILED: {e}")
        traceback.print_exc()

        err = {
            "crop": crop_path.name,
            "chart_type": chart_type,
            "error": str(e),
        }

        all_results.append(err)
        clean_results.append(err)

# ------------------------------------------------------------
# Save clean JSON only
# ------------------------------------------------------------
clean_json_path = DEMO_OUT / "bar_line_demo_results_clean.json"

with open(clean_json_path, "w") as f:
    json.dump(clean_results, f, indent=2, ensure_ascii=False)

ok = [r for r in clean_results if "error" not in r]
bars = [r for r in ok if r["chart_type"] == "bar"]
lines = [r for r in ok if r["chart_type"] == "line"]

print("\nDemo complete")
print("Clean JSON:", clean_json_path)
print("Reports:   ", DEMO_REPORT_DIR)
print(f"Processed: {len(ok)}/{len(target_df)} ({len(bars)} bar, {len(lines)} line)")

## 🎉 Done

Output structure:

```
/content/chart_demo_outputs/
├── pdf_pages/                              ← rendered PDF pages
├── chart_crops/                            ← detected chart images
├── page_chart_detection_overlays/          ← page-level detection viz
├── chart_crops_classified_metadata.csv     ← per-chart type predictions
└── bar_line_demo_outputs/
    ├── reports/                            ← per-chart combined overlay
    └── bar_line_demo_results.json          ← structured results
```



**To switch to a sample paper:** edit `PDF_PATH` in Step 3 to one of the files under `/content/chart_demo/sample_papers/`.